In [748]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    ExtraTreesRegressor,
    RandomForestRegressor,
    HistGradientBoostingRegressor
)

import joblib

import warnings
warnings.filterwarnings("ignore")

In [749]:
df= pd.read_csv("/content/drive/MyDrive/ALL DATASET/used_cars.csv")

In [750]:
df.head()

,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title,price
0,Ford,Utility Police Interceptor Base,2013,"51,000 mi.",E85 Flex Fuel,300.0HP 3.7L V6 Cylinder Engine Flex Fuel Capa...,6-Speed A/T,Black,Black,At least 1 accident or damage reported,Yes,"$10,300"
1,Hyundai,Palisade SEL,2021,"34,742 mi.",Gasoline,3.8L V6 24V GDI DOHC,8-Speed Automatic,Moonlight Cloud,Gray,At least 1 accident or damage reported,Yes,"$38,005"
2,Lexus,RX 350 RX 350,2022,"22,372 mi.",Gasoline,3.5 Liter DOHC,Automatic,Blue,Black,None reported,NaN,"$54,598"
3,INFINITI,Q50 Hybrid Sport,2015,"88,900 mi.",Hybrid,354.0HP 3.5L V6 Cylinder Engine Gas/Electric H...,7-Speed A/T,Black,Black,None reported,Yes,"$15,500"
4,Audi,Q3 45 S line Premium Plus,2021,"9,835 mi.",Gasoline,2.0L I4 16V GDI DOHC Turbo,8-Speed Automatic,Glacier White Metallic,Black,None reported,NaN,"$34,999"


In [751]:
df["milage"] = df["milage"].map(lambda x: int(x.strip().split(" ")[0].replace(',','')))

In [752]:
df["price"] = df["price"].map(lambda x:  int(x.strip().split("$")[-1].replace(",","")))

In [753]:
df["car_age"] = 2026 - df["model_year"]

In [754]:
def simplify_brand(x):
    economy = ['Ford','Toyota','Chevrolet','Nissan','Honda','Hyundai','Kia','Mazda']
    luxury = ['BMW','Mercedes-Benz','Audi','Lexus','Acura']
    exotic = ['Porsche','Ferrari','Lamborghini','Bentley','Rolls-Royce','McLaren']
    electric = ['Tesla','Rivian','Lucid']

    if x in economy:
        return 'Economy'
    elif x in luxury:
        return 'Luxury'
    elif x in exotic:
        return 'Exotic'



    elif x in electric:
        return 'Electric'
    else:
        return 'Other'
df['brand_group']=df['brand'].apply(simplify_brand)


In [755]:
import re

df["engine_size"] = (
    df["engine"].astype(str)
    .str.extract(r"(\d+(?:\.\d+)?)\s*L")[0]
    .astype(float)
)

def get_cylinders(x):
    x = str(x).upper()
    m = re.search(r'(?:V|I|W)(\d+)', x)
    if m:
        return int(m.group(1))
    m = re.search(r'(\d+)\s*(?:CYL|CYLINDER)', x)
    return int(m.group(1)) if m else np.nan

df["cylinders"] = df["engine"].apply(get_cylinders)


In [756]:
df["horsepower"] = (
    df["engine"]
    .astype(str)
    .str.extract(r"(\d+(?:\.\d+)?)\s*HP", expand=False)
    .astype(float)
)

In [757]:
def simplify_fuel(x):
    if x == 'Gasoline':
        return 'Gasoline'
    elif 'Hybrid' in str(x):
        return 'Hybrid'
    elif x == 'Diesel':
        return 'Diesel'
    else:
        return 'Other'

df['fuel_type'] = df['fuel_type'].apply(simplify_fuel)

In [758]:
df['horsepower'].fillna(df['horsepower'].median(), inplace=True)
df['engine_size'].fillna(df['engine_size'].median(), inplace=True)
df['cylinders'].fillna(df['cylinders'].median(), inplace=True)

In [759]:
def simplify_trans(x):
    x = str(x)

    if 'A/T' in x or 'Automatic' in x:
        return 'Automatic'

    elif 'M/T' in x or 'Manual' in x:
        return 'Manual'

    elif 'CVT' in x:
        return 'CVT'

    else:
        return 'Other'

df['transmission'] = df['transmission'].apply(simplify_trans)

In [760]:
df['accident'] = df['accident'].map({
    'None reported':0,
    'At least 1 accident or damage reported':1
})

In [761]:
df['accident'] = df['accident'].fillna(df['accident'].mode()[0])

In [762]:
df.drop(columns=["model_year","brand", "clean_title","model","engine","ext_col","int_col"], inplace=True)

In [763]:
df.head()

,milage,fuel_type,transmission,accident,price,car_age,brand_group,engine_size,cylinders,horsepower
0,51000,Other,Automatic,1.0,10300,13,Economy,3.7,6.0,300.0
1,34742,Gasoline,Automatic,1.0,38005,5,Economy,3.8,6.0,310.0
2,22372,Gasoline,Automatic,0.0,54598,4,Luxury,3.5,6.0,310.0
3,88900,Hybrid,Automatic,0.0,15500,11,Other,3.5,6.0,354.0
4,9835,Gasoline,Automatic,0.0,34999,5,Luxury,2.0,4.0,310.0


In [764]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4009 entries, 0 to 4008
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   milage        4009 non-null   int64  
 1   fuel_type     4009 non-null   object 
 2   transmission  4009 non-null   object 
 3   accident      4009 non-null   float64
 4   price         4009 non-null   int64  
 5   car_age       4009 non-null   int64  
 6   brand_group   4009 non-null   object 
 7   engine_size   4009 non-null   float64
 8   cylinders     4009 non-null   float64
 9   horsepower    4009 non-null   float64
dtypes: float64(4), int64(3), object(3)
memory usage: 313.3+ KB


In [765]:
# print("Shape:", df.shape)

# print("\nColumns:")
# print(df.columns.tolist())

# print("\nData types:")
# print(df.dtypes)

# print("\nMissing values:")
# print(df.isnull().sum())

# print("\nDuplicate rows:")
# print(df.duplicated().sum())

In [766]:
cat_cols = ['fuel_type','transmission','brand_group']
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)


In [767]:
df['milage'] = df['milage'].clip(
    df['milage'].quantile(0.01),
    df['milage'].quantile(0.99)
)

df['horsepower'] = df['horsepower'].clip(
    df['horsepower'].quantile(0.01),
    df['horsepower'].quantile(0.99)
)

In [768]:
df['price'] = np.log1p(df['price'])

q1 = df['price'].quantile(0.25)
q3 = df['price'].quantile(0.75)

iqr = q3 - q1

lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

df = df[(df['price'] >= lower) & (df['price'] <= upper)]


In [769]:
X = df.drop(columns=["price"])
y = df["price"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (3943, 16)
y shape: (3943,)


In [770]:
X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.2,random_state=42)


In [771]:
num_cols = ['milage','horsepower','engine_size','cylinders','car_age']
scaler = StandardScaler()

X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

In [772]:
df

,milage,accident,price,car_age,engine_size,cylinders,horsepower,fuel_type_Gasoline,fuel_type_Hybrid,fuel_type_Other,transmission_CVT,transmission_Manual,transmission_Other,brand_group_Electric,brand_group_Exotic,brand_group_Luxury,brand_group_Other
0,51000.0,1.0,9.239996,13,3.7,6.0,300.0,False,False,True,False,False,False,False,False,False,False
1,34742.0,1.0,10.545499,5,3.8,6.0,310.0,True,False,False,False,False,False,False,False,False,False
2,22372.0,0.0,10.907771,4,3.5,6.0,310.0,True,False,False,False,False,False,False,False,True,False
3,88900.0,0.0,9.648660,11,3.5,6.0,354.0,False,True,False,False,False,False,False,False,False,True
4,9835.0,0.0,10.463103,5,2.0,4.0,310.0,True,False,False,False,False,False,False,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4003,53705.0,1.0,10.162037,8,2.0,4.0,241.0,True,False,False,False,False,False,False,False,True,False
4005,10900.0,0.0,10.894904,4,3.0,6.0,349.0,True,False,False,False,False,True,False,False,True,False
4006,2116.0,0.0,11.418604,4,3.5,6.0,310.0,False,False,True,False,False,False,False,True,False,False
4007,33000.0,0.0,11.050890,6,3.5,6.0,450.0,True,False,False,False,False,False,False,False,False,False


In [773]:
pd.set_option('display.max_columns', None)



In [774]:
X_train.head()
# milage-----
# accident----
# car age------
# engine_size -----
# cylinders----
# horsepower----
# fuel type -> gasoline, hybrid, other, Diesel------
# transmission -> CVT, Manual, Other, Automatic------
# brand_group -> Electric, Exotic, Luxury, Other( come from brand) -------


,milage,accident,car_age,engine_size,cylinders,horsepower,fuel_type_Gasoline,fuel_type_Hybrid,fuel_type_Other,transmission_CVT,transmission_Manual,transmission_Other,brand_group_Electric,brand_group_Exotic,brand_group_Luxury,brand_group_Other
3390,-0.086558,0.0,1.076751,1.368887,1.168472,0.537383,True,False,False,False,False,False,False,False,True,False
1139,-1.175447,0.0,-1.060902,-0.108105,-0.105197,-0.060649,False,False,True,False,False,False,False,False,False,False
1545,0.123814,0.0,-0.403162,-0.108105,-0.105197,-0.176397,False,True,False,True,False,False,False,False,True,False
2840,-1.004471,0.0,-1.060902,-0.551203,-0.105197,-0.012421,True,False,False,False,False,False,False,True,False,False
2975,0.679683,1.0,0.912316,-0.846601,-0.742031,-1.507501,True,False,False,False,False,False,False,False,False,True


In [775]:
model = LinearRegression()
model.fit(X_train, y_train)
pred = model.predict(X_test)

In [776]:
print("R2 Score:", r2_score(y_test, pred))
print("MAE:", mean_absolute_error(y_test, pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, pred)))

R2 Score: 0.8069993519380037
MAE: 0.2572675702256098
RMSE: 0.34803538023489056


In [777]:
import pickle


with open("model.pkl","wb") as file:
  pickle.dump(model, file)

In [811]:
feature_columns = X_train.columns.tolist()

with open("feature_columns.pkl", "wb") as f:
    pickle.dump(feature_columns, f)

In [812]:
with open("scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

In [816]:
#-----------------------------
#
#-----------------------------
#
#----------------------------

In [819]:
import pickle

with open("model.pkl", "rb") as f:
    m = pickle.load(f)

with open("scaler.pkl", "rb") as f:
    scaler = pickle.load(f)

with open("feature_columns.pkl", "rb") as f:
    feature_columns = pickle.load(f)

In [820]:
import pandas as pd
import numpy as np
import pickle
import re


def simplify_brand(x):
    economy = ['Ford', 'Toyota', 'Chevrolet', 'Nissan','Honda', 'Hyundai', 'Kia', 'Mazda']
    luxury = ['BMW', 'Mercedes-Benz', 'Audi', 'Lexus', 'Acura']
    exotic = ['Porsche', 'Ferrari', 'Lamborghini','Bentley', 'Rolls-Royce', 'McLaren']
    electric = ['Tesla', 'Rivian', 'Lucid']

    if x in economy:
        return 'Economy'
    elif x in luxury:
        return 'Luxury'
    elif x in exotic:
        return 'Exotic'
    elif x in electric:
        return 'Electric'
    else:
        return 'Other'


def predict_car_price(brand, model_year, milage, fuel_type, engine_size, horsepower, cylinders, transmission, ext_col, int_col, accident):


    # Create dataframe
    car = pd.DataFrame([{
        'milage': milage,
        'accident': accident,
        'brand': brand,
        'model_year': model_year,
        'fuel_type': fuel_type,
        'engine_size': engine_size,
        'horsepower': horsepower,
        'cylinders': cylinders,
        'transmission': transmission,
        'ext_col': ext_col,
        'int_col': int_col,

    }])


    car['car_age'] = 2026 - car['model_year']

    # Brand group
    car['brand_group'] = car['brand'].apply(simplify_brand)


    # Accident
    if accident == 'No':
        car['accident'] = 0
    else:
        car['accident'] = 1


    # Remove unused columns
    car.drop( columns=[ 'model_year', 'brand'], inplace=True )


    # One-hot encoding
    cat_cols = ['fuel_type', 'transmission', 'brand_group' ]
    car = pd.get_dummies( car, columns=cat_cols, drop_first=True )

    # X_train columns from your training data
    car = car.reindex(
        columns= feature_columns,
        fill_value=False
    )


    # Scale numerical columns
    num_cols = [ 'milage', 'horsepower', 'engine_size', 'cylinders', 'car_age' ]
    car[num_cols] = scaler.transform( car[num_cols] )


    # Predict
    log_price = m.predict(car)[0]

    # Reverse log1p
    price = np.expm1(log_price)

    return price

In [823]:
price = predict_car_price(
    brand="Ford",
    model_year=2024,
    milage=51000,
    fuel_type="Gasoline",
    engine_size = 3.7,
    horsepower = 600.0,
    cylinders = 6,
    transmission="Manual",
    ext_col="Black",
    int_col="Black",
    accident="No"
)

print(f"Predicted car price: ${price:,.2f}")

Predicted car price: $111,059.80
